# CivicPulse AI — Qwen3-VL free-GPU prototype

Prototype only. Runs interactively in a free Colab GPU session; it is not an always-on API. Do not upload real citizen/private images unless you have authorization and have reviewed the notebook/runtime privacy implications. Use synthetic, public, or consented test images.

Model: `Qwen/Qwen3-VL-4B-Instruct` (Apache-2.0). The notebook uses 4-bit loading to reduce VRAM use. GPU availability and runtime limits are controlled by the notebook provider.

In [13]:
!pip -q install -U "transformers>=4.57.0" accelerate bitsandbytes qwen-vl-utils
# Force a clean and compatible Pillow reinstall to fix Python 3.13 library mismatch
!pip -q install -U --force-reinstall pillow

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


In [14]:
import torch
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration
from qwen_vl_utils import process_vision_info

if not torch.cuda.is_available():
    raise RuntimeError("No GPU is attached. In Colab choose Runtime > Change runtime type > T4 GPU, if available.")

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)

GPU: Tesla T4
CUDA: 13.0


In [15]:
MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model.eval()
print("Model loaded:", MODEL_ID)

Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

Model loaded: Qwen/Qwen3-VL-4B-Instruct


## Upload a test image

Use a single test photo showing a possible pothole, garbage pile, sewage issue, or damaged streetlight.

In [22]:
from google.colab import files
uploaded = files.upload()
if not uploaded:
    raise RuntimeError("Upload one test image first.")
image_path = next(iter(uploaded))
print("Selected:", image_path)

Saving istockphoto-876880612-612x612.jpg to istockphoto-876880612-612x612.jpg
Selected: istockphoto-876880612-612x612.jpg


In [23]:
import json, re
import torch

SYSTEM_PROMPT = """You are an image-triage assistant for a civic issue reporting system.
Analyze only visible evidence. Do not infer an exact address, identity, cause, or hidden facts.
Return one JSON object and no markdown with these keys:
{
  "defect_type": "pothole|garbage|sewage|streetlight|other|unclear",
  "confidence": 0.0,
  "severity": "low|medium|high|unclear",
  "safety_risk": "low|medium|high|unclear",
  "summary": "short description of visible evidence",
  "evidence": ["observable visual detail"],
  "recommended_action": "suggested municipal inspection/action",
  "human_review_required": true
}
Be conservative. If the image is unclear or does not show a civic defect, use defect_type=unclear or other and low confidence. Always set human_review_required=true. Confidence is an estimate, not a verified probability."""

messages = [{
    "role": "user",
    "content": [
        {
            "type": "image",
            "image": image_path,
            "min_pixels": 256 * 256,
            "max_pixels": 360 * 360  # Lowers resolution to avoid CUDA OOM
        },
        {"type": "text", "text": SYSTEM_PROMPT + "\nAnalyze this image for civic issue triage."},
    ],
}]

# Note: Make sure cell _10P3bnUqmuI has been executed to define 'processor' and 'model'
prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[prompt],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
).to(model.device)

with torch.inference_mode():
    generated_ids = model.generate(**inputs, max_new_tokens=400, do_sample=False)

trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated_ids)]
raw = processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
print(raw)

{
  "defect_type": "sewage",
  "confidence": 0.95,
  "severity": "high",
  "safety_risk": "high",
  "summary": "Dark, murky sewage is flowing from two open pipes into a body of water, creating a foamy, turbulent surface.",
  "evidence": [
    "Two open pipes discharging dark, viscous liquid",
    "Foamy, turbulent water at the discharge point",
    "Sewage-like material flowing into water body",
    "Exposed earth bank with no visible containment or treatment structure"
  ],
  "recommended_action": "Immediate inspection and closure of discharge pipes; assess environmental and public health impact.",
  "human_review_required": true
}


In [24]:
# Best-effort JSON extraction for evaluation; inspect raw output if parsing fails.
match = re.search(r"\{.*\}", raw, flags=re.S)
if not match:
    raise ValueError("The model did not return a JSON object. Inspect the raw output above.")
result = json.loads(match.group(0))
print(json.dumps(result, indent=2, ensure_ascii=False))

assert result.get("human_review_required") is True, "Human review must remain enabled."
assert result.get("defect_type") in {"pothole", "garbage", "sewage", "streetlight", "other", "unclear"}

{
  "defect_type": "sewage",
  "confidence": 0.95,
  "severity": "high",
  "safety_risk": "high",
  "summary": "Dark, murky sewage is flowing from two open pipes into a body of water, creating a foamy, turbulent surface.",
  "evidence": [
    "Two open pipes discharging dark, viscous liquid",
    "Foamy, turbulent water at the discharge point",
    "Sewage-like material flowing into water body",
    "Exposed earth bank with no visible containment or treatment structure"
  ],
  "recommended_action": "Immediate inspection and closure of discharge pipes; assess environmental and public health impact.",
  "human_review_required": true
}


## Before integrating with CivicPulse

- Evaluate on a labelled set covering all supported defect categories, normal scenes, night/rain/blurry photos, and misleading images.
- Measure per-class precision/recall, confusion matrix, JSON validity, latency, and failure rate. Do not choose a production threshold from a single image.
- Keep server-side priority rules and human municipal review. The model must not autonomously reject reports or declare repairs complete.
- This notebook does not expose a public endpoint, store images in CivicPulse, or change the existing Render/Supabase deployment.
- For a ₹0 setup, run it manually when a GPU is available. Do not keep a notebook alive as a production web server; free runtimes can stop without notice.